# **Env & Imports**

In [ ]:
##!pip -q install -U transformers datasets accelerate evaluate scikit-learn huggingface_hub fsspec


In [2]:

import os, random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
import evaluate

# Reproducibility
SEED = 555
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


# **Choosing Backbone: XLM-RoBERTa**

In [3]:
# Choosing Backbone: XLM-RoBERTa

MODEL_BACKBONE = "xlm-roberta-base"
NUM_LABELS     = 2                       # set to your number of classes
MAX_LEN        = 128                     # typical 128/256; raise if texts are long
BATCH_SIZE     = 16
N_EPOCHS       = 3
LR             = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_BACKBONE, use_fast=True)
config = AutoConfig.from_pretrained(MODEL_BACKBONE, num_labels=NUM_LABELS)
print("Special tokens map:", tokenizer.special_tokens_map)
print("Vocab size:", tokenizer.vocab_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Special tokens map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
Vocab size: 250002


# **Tokenization Basics (Quick Demo)**

- `input_ids`: token IDs
- `attention_mask`: 1 for real tokens, 0 for padding
- `tokenizer.encode_plus()` supports **single** and **pair** sentences
- `tokenizer.decode()` converts IDs back to text

In [4]:
# Single sentence
demo = tokenizer.encode_plus(
    "Transformers are powerful models for NLP.",
    max_length=MAX_LEN, truncation=True, padding="max_length", return_tensors="pt"
)
print("Single sentence shapes:", {k: v.shape for k, v in demo.items()})

# Two sentences (pair)
demo_pair = tokenizer.encode_plus(
    "Ada mailed the package.", "Ben received it Monday.",
    max_length=MAX_LEN, truncation=True, padding="max_length", return_tensors="pt"
)
print("Sentence pair shapes:", {k: v.shape for k, v in demo_pair.items()})

# Decode the first few token IDs (without padding)
ids = demo["input_ids"][0]
mask = demo["attention_mask"][0].bool()
print("Decoded:", tokenizer.decode(ids[mask]))

Single sentence shapes: {'input_ids': torch.Size([1, 128]), 'attention_mask': torch.Size([1, 128])}
Sentence pair shapes: {'input_ids': torch.Size([1, 128]), 'attention_mask': torch.Size([1, 128])}
Decoded: <s> Transformers are powerful models for NLP.</s>


## Load & Inspect Dataset

> Expected columns: `text`, `label`. If `label` is string, we auto-map to integers.

In [6]:
TRAIN_CSV = "train.csv"
TEST_CSV  = "test.csv"

train_df = pd.read_csv(TRAIN_CSV)
print("Train shape:", train_df.shape)
display(train_df.head())

if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    print("Test shape:", test_df.shape)
    display(test_df.head())
else:
    test_df = None

# Create 'text' column by concatenating 'premise' and 'hypothesis'
# using the tokenizer's separator token for paired inputs.
if "premise" in train_df.columns and "hypothesis" in train_df.columns:
    train_df["text"] = train_df["premise"] + " " + tokenizer.sep_token + " " + train_df["hypothesis"]
    if test_df is not None and "premise" in test_df.columns and "hypothesis" in test_df.columns:
        test_df["text"] = test_df["premise"] + " " + tokenizer.sep_token + " " + test_df["hypothesis"]
elif "premise" in train_df.columns: # Fallback if only premise is available
    train_df["text"] = train_df["premise"]
    if test_df is not None and "premise" in test_df.columns:
        test_df["text"] = test_df["premise"]
elif "hypothesis" in train_df.columns: # Fallback if only hypothesis is available
    train_df["text"] = train_df["hypothesis"]
    if test_df is not None and "hypothesis" in test_df.columns:
        test_df["text"] = test_df["hypothesis"]

# Ensure required columns exist after creating 'text' column
assert "text" in train_df.columns and "label" in train_df.columns, "Expected columns: text, label"

# Map string labels -> ints (and keep a reverse map)
if train_df["label"].dtype == "O":
    label2id = {lbl: i for i, lbl in enumerate(sorted(train_df["label"].unique()))}
    id2label = {i: lbl for lbl, i in label2id.items()}
    train_df["label"] = train_df["label"].map(label2id)
else:
    # numeric already
    uniq = sorted(train_df["label"].unique().tolist())
    id2label = {int(i): int(i) for i in uniq}
    label2id = {int(i): int(i) for i in uniq}

NUM_LABELS = len(id2label)
print("Labels:", id2label)

Train shape: (12120, 6)


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


Test shape: (5195, 5)


,id,premise,hypothesis,lang_abv,language
0,c6d58c3f69,بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولم...,"کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی...",ur,Urdu
1,cefcc82292,هذا هو ما تم نصحنا به.,عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت ال...,ar,Arabic
2,e98005252c,et cela est en grande partie dû au fait que le...,Les mères se droguent.,fr,French
3,58518c10ba,与城市及其他公民及社区组织代表就IMA的艺术发展进行对话&amp,IMA与其他组织合作，因为它们都依靠共享资金。,zh,Chinese
4,c32b0d16df,Она все еще была там.,"Мы думали, что она ушла, однако, она осталась.",ru,Russian


Labels: {0: 0, 1: 1, 2: 2}


## Preprocessing: Tokenize → HF Datasets

We build a tokenize function and apply it to a Hugging Face `Dataset`.

In [7]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

hf_full = Dataset.from_pandas(train_df, preserve_index=False)
hf_full = hf_full.map(tokenize_batch, batched=True)
cols_to_keep = ["input_ids", "attention_mask", "label"]
hf_full.set_format(type="torch", columns=cols_to_keep)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=None)

Map:   0%|          | 0/12120 [00:00<?, ? examples/s]

## Metrics & Trainer Helpers
We’ll report **accuracy** and **macro-F1** per fold.

In [8]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": f1}

## 5-Fold Stratified Cross-Validation

- Keeps label distribution consistent across folds.
- Trains and evaluates a fresh model on each fold.
- Reports metrics per fold and overall mean.

In [12]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

all_fold_metrics = []
fold_idx = 0

for train_index, valid_index in skf.split(train_df["text"], train_df["label"]):
    fold_idx += 1
    print(f"\n===== Fold {fold_idx} / 5 =====")
    train_split = hf_full.select(train_index.tolist())
    valid_split = hf_full.select(valid_index.tolist())

    # Fresh model per fold
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_BACKBONE, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
    ).to(device)

    out_dir = f"/content/model_{os.path.basename(MODEL_BACKBONE)}_fold{fold_idx}"

    args = TrainingArguments(
        output_dir=out_dir,
        evaluation_strategy="epoch", # Changed 'eval' to 'evaluation_strategy'
        save_strategy="epoch",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=N_EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_steps=50,
        report_to="none",   # set to "wandb" or "tensorboard" if you use them
        seed=SEED
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_split,
        eval_dataset=valid_split,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()
    print("Fold metrics:", metrics)
    all_fold_metrics.append(metrics)

# Aggregate metrics across folds
df_metrics = pd.DataFrame(all_fold_metrics)[["eval_loss", "eval_accuracy", "eval_macro_f1"]]
print("\n===== Cross-Validation Summary =====")
display(df_metrics)
print("Mean metrics:")
display(df_metrics.mean().to_frame("mean").T)


===== Fold 1 / 5 =====


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## Train on Full Data & Predict on Test Set

If you have a `test.csv`, we’ll train a final model on **all** training data and predict on the test set.

In [ ]:
if test_df is not None:
    # Tokenize test
    hf_test = Dataset.from_pandas(test_df, preserve_index=False)
    hf_test = hf_test.map(tokenize_batch, batched=True)
    hf_test.set_format(type="torch", columns=["input_ids", "attention_mask"])

    # Train final model on full train
    final_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_BACKBONE, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
    ).to(device)

    final_args = TrainingArguments(
        output_dir=f"/content/final_{os.path.basename(MODEL_BACKBONE)}",
        evaluation_strategy="no",
        save_strategy="no",
        learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE,
        num_train_epochs=N_EPOCHS,
        weight_decay=0.01,
        report_to="none",
        seed=SEED
    )

    final_trainer = Trainer(
        model=final_model,
        args=final_args,
        train_dataset=hf_full,
        tokenizer=tokenizer,
        data_collator=data_collator
    )
    final_trainer.train()

    # Predict
    raw_preds = final_trainer.predict(hf_test).predictions
    y_pred = raw_preds.argmax(axis=-1)

    # Map back to original label names if needed
    if isinstance(next(iter(id2label.values())), str):
        y_pred = [id2label[int(i)] for i in y_pred]

    # Save submission
    pred_path = "/content/predictions.csv"
    pd.DataFrame({"prediction": y_pred}).to_csv(pred_path, index=False)
    print("Saved predictions to:", pred_path)